In [19]:
import pandas as pd
from tabpfn import TabPFNClassifier, TabPFNRegressor
from tabpfn_extensions.embedding import TabPFNEmbedding
import numpy as np
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedShuffleSplit
load_dotenv(".env")

True

In [20]:
from benchmark_datasets import OpenMLBenchmark

In [21]:
bench = OpenMLBenchmark()
print("Benchmark suite (20 datasets):")
print(bench.list().to_string(index=False))

ds = bench.load("lymph")
print(f"\nLoaded {ds.name!r}: X={ds.X.shape} ({ds.X.dtype}), y={ds.y.shape}")
print(f"classes: {ds.n_classes}, feature count: {len(ds.feature_names)}")

Benchmark suite (20 datasets):
                  name  data_id           task  n_rows                                                                          note
           tic-tac-toe       50 classification     958                                       XOR-like win patterns, pure interaction
      monks-problems-2      334 classification     601                                                 synthetic XOR / parity target
                 sonar       40 classification     208                                60 correlated sonar bands, non-linear boundary
            ionosphere       59 classification     351                                                      radar signal, non-linear
               vehicle       54 classification     846                                                   4-class silhouette geometry
                  wdbc     1510 classification     569                                breast cancer, non-linear feature interactions
              diabetes       37 classi

In [22]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neural_network import MLPRegressor
from aug_dataset import aug_dataset

In [23]:
from sklearn.metrics import accuracy_score
sss = StratifiedShuffleSplit(1,train_size=64)
train_index, test_index = next(sss.split(ds.X, ds.y))
X_train = ds.X[train_index]
y_train = ds.y[train_index]
X_test = ds.X[test_index]
y_test = ds.y[test_index]

print('Num labeled samples:', len(X_train))

clf = TabPFNClassifier()
clf.fit(X_train, y_train)
base_pfn_pred = clf.predict(X_test)
print('Base tabPFN acc: ', accuracy_score(y_test,base_pfn_pred))

X_aug = aug_dataset(3, X_train, 0.8, 1, 0.5) 

X_aug_conf_indx = abs(0.5-clf.predict_proba(X_aug)[:, 0])>0.45
X_aug_conf = X_aug[X_aug_conf_indx]
print('Num augmented samples:', len(X_aug))
print('Num confident augmented samples:', len(X_aug_conf))

y_aug = clf.predict(X_aug)
y_aug_conf = clf.predict(X_aug_conf)

y_sl_aug = clf.predict_proba(X_aug)[:,1]

print('done')

Num labeled samples: 64
Base tabPFN acc:  0.8809523809523809
Num augmented samples: 256
Num confident augmented samples: 217
done


In [24]:
results = []

for n_estim in [1,2,3,5,10,100]:
    rfr = RandomForestClassifier(n_estimators=n_estim)
    rfr.fit(X_train, y_train)
    base_rfr_pred = rfr.predict(X_test)
    print(F'RFR (n_estim={n_estim}) base acc: ', accuracy_score(y_test,base_rfr_pred))
    

    rfr_aug = RandomForestClassifier(n_estimators=n_estim)
    rfr_aug.fit(X_aug, y_aug)
    rfr_aug_pred = rfr_aug.predict(X_test)
    print(F'RFR (n_estim={n_estim}) aug acc: ', accuracy_score(y_test,rfr_aug_pred))

    rfr_aug_conf = RandomForestClassifier(n_estimators=n_estim)
    rfr_aug_conf.fit(X_aug_conf, y_aug_conf)
    rfr_aug_conf_pred = rfr_aug_conf.predict(X_test)
    print(F'RFR (n_estim={n_estim}) aug conf acc: ', accuracy_score(y_test,rfr_aug_conf_pred))

    results.append({'name': f'RFR n_estim={n_estim}', 
                    'base': accuracy_score(y_test,base_rfr_pred),
                    'aug': accuracy_score(y_test, rfr_aug_pred),
                    'aug confident': accuracy_score(y_test, rfr_aug_conf_pred),
                    'solf-labels aug': None})

for hid_size in [(), (8,),(32,),(128,), (16,16,), (32,32), (16,16,16,), (67,67,67,)]:
    mlp = MLPClassifier(hidden_layer_sizes=hid_size, max_iter=5000)
    mlp.fit(X_train, y_train)
    base_mlp_pred = mlp.predict(X_test)
    print(f'MLP (hidden_l={hid_size}) base acc: ', accuracy_score(y_test,base_mlp_pred))

    mlp_aug = MLPClassifier(hidden_layer_sizes=hid_size, max_iter=5000)
    mlp_aug.fit(X_aug, y_aug)
    mlp_aug_pred= mlp_aug.predict(X_test)
    print(f'MLP (hidden_l={hid_size}) aug acc: ', accuracy_score(y_test,mlp_aug_pred))

    mlp_aug_conf = MLPClassifier(hidden_layer_sizes=hid_size, max_iter=5000)
    mlp_aug_conf.fit(X_aug_conf, y_aug_conf)
    mlp_aug_conf_pred= mlp_aug_conf.predict(X_test)
    print(f'MLP (hidden_l={hid_size}) aug conf acc: ', accuracy_score(y_test,mlp_aug_conf_pred))

    mlp_sl_aug = MLPRegressor(hidden_layer_sizes=hid_size, max_iter=5000)
    mlp_sl_aug.fit(X_aug, y_sl_aug)
    mlp_sl_aug_pred = mlp_sl_aug.predict(X_test) > 0.5
    print(f'MLP (hidden_l={hid_size}) soft-labels aug acc: ', accuracy_score(y_test,mlp_sl_aug_pred))

    results.append({'name': f'MLP hidden_l={hid_size}', 
                'base': accuracy_score(y_test,base_mlp_pred),
                'aug': accuracy_score(y_test, mlp_aug_pred),
                'aug confident': accuracy_score(y_test, mlp_aug_conf_pred),
                'solf-labels aug': accuracy_score(y_test, mlp_sl_aug_pred)})

results_df = pd.DataFrame(results)
results_df

RFR (n_estim=1) base acc:  0.6547619047619048
RFR (n_estim=1) aug acc:  0.7738095238095238
RFR (n_estim=1) aug conf acc:  0.5952380952380952
RFR (n_estim=2) base acc:  0.7738095238095238
RFR (n_estim=2) aug acc:  0.7976190476190477
RFR (n_estim=2) aug conf acc:  0.7976190476190477
RFR (n_estim=3) base acc:  0.8214285714285714
RFR (n_estim=3) aug acc:  0.7380952380952381
RFR (n_estim=3) aug conf acc:  0.7857142857142857
RFR (n_estim=5) base acc:  0.8333333333333334
RFR (n_estim=5) aug acc:  0.8214285714285714
RFR (n_estim=5) aug conf acc:  0.8452380952380952
RFR (n_estim=10) base acc:  0.8333333333333334
RFR (n_estim=10) aug acc:  0.8214285714285714
RFR (n_estim=10) aug conf acc:  0.8214285714285714
RFR (n_estim=100) base acc:  0.8452380952380952
RFR (n_estim=100) aug acc:  0.8095238095238095
RFR (n_estim=100) aug conf acc:  0.8333333333333334
MLP (hidden_l=()) base acc:  0.8571428571428571
MLP (hidden_l=()) aug acc:  0.8214285714285714
MLP (hidden_l=()) aug conf acc:  0.821428571428571

,name,base,aug,aug confident,solf-labels aug
0,RFR n_estim=1,0.654762,0.773810,0.595238,NaN
1,RFR n_estim=2,0.773810,0.797619,0.797619,NaN
2,RFR n_estim=3,0.821429,0.738095,0.785714,NaN
3,RFR n_estim=5,0.833333,0.821429,0.845238,NaN
4,RFR n_estim=10,0.833333,0.821429,0.821429,NaN
5,RFR n_estim=100,0.845238,0.809524,0.833333,NaN
6,MLP hidden_l=(),0.857143,0.821429,0.821429,0.357143
7,"MLP hidden_l=(8,)",0.833333,0.809524,0.809524,0.380952
8,"MLP hidden_l=(32,)",0.833333,0.809524,0.797619,0.369048
9,"MLP hidden_l=(128,)",0.845238,0.821429,0.797619,0.380952


## Evaluate every classification dataset

The single-dataset exploration above is generalized below into a sweep that runs across **every**
classification dataset in the benchmark suite over repeated stratified splits, then aggregates the
results -- the same evaluation structure as in `mlp_distillation.ipynb`.

Per dataset we compare the TabPFN teacher against Random-Forest and MLP students trained four ways:

- **base** -- trained on the few real training rows with their true labels.
- **aug** -- trained on `aug_dataset` synthetic rows, hard-labeled by the TabPFN teacher.
- **aug confident** -- the same synthetic rows, kept only where the teacher is confident
  (`predict_proba.max(axis=1) > 0.95`, the multiclass generalization of the binary `|0.5 - p| > 0.45`
  filter used above).
- **soft labels** -- (MLP only) an `MLPRegressor` fit on the teacher's full `predict_proba` matrix
  over the synthetic rows, with the predicted class taken as the argmax. This generalizes the binary
  single-column soft-label regression used above to any number of classes.

Both **accuracy** and **ROC AUC** are reported (AUC is one-vs-rest macro for multiclass via
`benchmark_datasets._roc_auc`), aggregated into `per_dataset` and `summary` DataFrames. Features are
**not** standardized here because `aug_dataset` relies on raw 0/1 values to detect one-hot columns.


In [25]:
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier, MLPRegressor
from benchmark_datasets import _roc_auc   # binary positive-class / multiclass one-vs-rest macro AUC
from aug_dataset import aug_dataset

MODEL_NAMES = [
    "tabpfn",
    "rf_base", "rf_aug", "rf_aug_conf",
    "mlp_base", "mlp_aug", "mlp_aug_conf", "mlp_soft",
]

# student factories (fresh estimator per fit so nothing leaks across splits)
def _make_rf():
    return RandomForestClassifier(n_estimators=100)

def _make_mlp():
    return MLPClassifier(hidden_layer_sizes=(), max_iter=5000)

def _make_mlp_reg():
    return MLPRegressor(hidden_layer_sizes=(), max_iter=5000)


def _align_proba(proba, clf_classes, classes):
    """Re-map a classifier's probability columns onto the full ``classes`` ordering.

    A student fitted on a confidence-filtered subset may not have seen every class; its
    ``predict_proba`` then has fewer/reordered columns. ROC AUC (one-vs-rest macro) needs one column
    per class in a fixed order, so missing classes get a zero column.
    """
    proba = np.asarray(proba)
    if proba.shape[1] == len(classes) and np.array_equal(clf_classes, classes):
        return proba
    idx = {c: i for i, c in enumerate(classes)}
    full = np.zeros((proba.shape[0], len(classes)))
    for j, c in enumerate(clf_classes):
        full[:, idx[c]] = proba[:, j]
    return full


def _fit_score_clf(make_clf, X_fit, y_fit, X_test, y_test, classes):
    """Fit a classifier and return ``(accuracy, roc_auc)`` on the test fold.

    Returns ``(nan, nan)`` when the fit set is degenerate (fewer than 2 classes), e.g. an
    over-aggressive confidence filter that kept a single class.
    """
    if len(np.unique(y_fit)) < 2:
        return float("nan"), float("nan")
    clf = make_clf()
    clf.fit(X_fit, y_fit)
    acc = accuracy_score(y_test, clf.predict(X_test))
    auc = _roc_auc(y_test, _align_proba(clf.predict_proba(X_test), clf.classes_, classes), classes)
    return acc, auc


def _fit_score_soft(make_reg, X_fit, soft_targets, teacher_classes, X_test, y_test, classes):
    """Soft-label student: regress the teacher's full ``predict_proba`` matrix, predict by argmax.

    Generalizes the binary single-column soft-label regression to any number of classes (the
    regressor is multi-output, one column per teacher class).
    """
    reg = make_reg()
    reg.fit(X_fit, soft_targets)
    out = np.atleast_2d(reg.predict(X_test))
    pred = teacher_classes[out.argmax(axis=1)]
    acc = accuracy_score(y_test, pred)
    auc = _roc_auc(y_test, _align_proba(out, teacher_classes, classes), classes)
    return acc, auc


def evaluate_models_on_dataset(
    ds, *, train_size=4, n_splits=5, augment=16, conf_threshold=0.95,
):
    """Run the synthetic-label / augmentation comparison across repeated splits of one dataset.

    For each split a TabPFN teacher is fitted on the few real training rows, then RF and MLP students
    are trained four ways: on the real rows (``base``), on ``aug_dataset`` synthetic rows hard-labeled
    by the teacher (``aug``), on the confidently-labeled subset of those rows
    (``aug_conf``, ``predict_proba.max > conf_threshold``), and -- for the MLP -- on the teacher's full
    soft-probability matrix over the synthetic rows (``mlp_soft``). Features are left unscaled because
    ``aug_dataset`` detects one-hot columns from raw 0/1 values. Returns
    ``{model: {"acc": [...], "auc": [...]}}`` with one entry per split.
    """
    X, y = ds.X, ds.y
    classes = np.unique(y)
    res = {m: {"acc": [], "auc": []} for m in MODEL_NAMES}

    for train_idx, test_idx in bench.splits(
        X, y, task="classification", n_splits=n_splits, train_size=train_size
    ):
        X_train, y_train = X[train_idx], y[train_idx]
        X_test, y_test = X[test_idx], y[test_idx]

        # --- Teacher: TabPFN ---
        tpfn = TabPFNClassifier().fit(X_train, y_train)
        res["tabpfn"]["acc"].append(accuracy_score(y_test, tpfn.predict(X_test)))
        res["tabpfn"]["auc"].append(
            _roc_auc(y_test, _align_proba(tpfn.predict_proba(X_test), tpfn.classes_, classes), classes)
        )

        # --- Synthetic rows, hard- and soft-labeled by the teacher ---
        X_aug = aug_dataset(augment, X_train)
        aug_proba = tpfn.predict_proba(X_aug)
        y_aug = tpfn.classes_[aug_proba.argmax(axis=1)]
        conf = aug_proba.max(axis=1) > conf_threshold
        X_aug_conf, y_aug_conf = X_aug[conf], y_aug[conf]

        # --- Random-Forest students ---
        for name, (Xf, yf) in {
            "rf_base": (X_train, y_train),
            "rf_aug": (X_aug, y_aug),
            "rf_aug_conf": (X_aug_conf, y_aug_conf),
        }.items():
            acc, auc = _fit_score_clf(_make_rf, Xf, yf, X_test, y_test, classes)
            res[name]["acc"].append(acc)
            res[name]["auc"].append(auc)

        # --- MLP students ---
        for name, (Xf, yf) in {
            "mlp_base": (X_train, y_train),
            "mlp_aug": (X_aug, y_aug),
            "mlp_aug_conf": (X_aug_conf, y_aug_conf),
        }.items():
            acc, auc = _fit_score_clf(_make_mlp, Xf, yf, X_test, y_test, classes)
            res[name]["acc"].append(acc)
            res[name]["auc"].append(auc)

        acc, auc = _fit_score_soft(
            _make_mlp_reg, X_aug, aug_proba, tpfn.classes_, X_test, y_test, classes
        )
        res["mlp_soft"]["acc"].append(acc)
        res["mlp_soft"]["auc"].append(auc)

    return res


def evaluate_all_classification(datasets=None, *, verbose=True, **kwargs):
    """Evaluate every model over every classification dataset and aggregate.

    Args:
        datasets: optional subset of dataset names (defaults to the whole suite).
        verbose: print a per-dataset line as each finishes.
        **kwargs: forwarded to ``evaluate_models_on_dataset`` (``train_size``, ``n_splits``,
            ``augment``, ``conf_threshold``).

    Returns:
        ``(per_dataset, summary)`` DataFrames. ``per_dataset`` has acc/auc columns per model
        (averaged over splits); ``summary`` averages each model across all datasets.
    """
    specs = [s for s in bench.specs if s.task == "classification"]
    if datasets is not None:
        wanted = set(datasets)
        specs = [s for s in specs if s.name in wanted]

    rows = []
    for spec in specs:
        ds = bench.load(spec)
        try:
            res = evaluate_models_on_dataset(ds, **kwargs)
        except Exception as exc:                       # keep the sweep going on a bad dataset
            print(f"[skip] {spec.name}: {exc}")
            continue
        row = {"dataset": spec.name, "n_rows": spec.n_rows, "n_classes": ds.n_classes}
        for m in MODEL_NAMES:
            row[f"{m}_acc"] = float(np.nanmean(res[m]["acc"]))
            row[f"{m}_auc"] = float(np.nanmean(res[m]["auc"]))
        rows.append(row)
        if verbose:
            cells = " | ".join(f"{m}={row[f'{m}_acc']:.3f}" for m in MODEL_NAMES)
            print(f"{spec.name:24s} (k={ds.n_classes}) {cells}")

    per_dataset = pd.DataFrame(rows)
    summary = pd.DataFrame(
        {
            "model": MODEL_NAMES,
            "acc_mean": [per_dataset[f"{m}_acc"].mean() for m in MODEL_NAMES],
            "acc_std": [per_dataset[f"{m}_acc"].std() for m in MODEL_NAMES],
            "auc_mean": [per_dataset[f"{m}_auc"].mean() for m in MODEL_NAMES],
            "auc_std": [per_dataset[f"{m}_auc"].std() for m in MODEL_NAMES],
        }
    )
    return per_dataset, summary

In [26]:
# Run the full sweep over every classification dataset, then aggregate.
# Heavy: ~50 datasets x n_splits x (TabPFN fit + augment + RF/MLP students).
# Pass e.g. datasets=["lymph", "wine"] or a smaller n_splits for a quick smoke test.
per_dataset, summary = evaluate_all_classification()

print("\n=== Aggregate across all classification datasets ===")
display(summary.round(3))
display(per_dataset.round(3))

# Mean ROC AUC across all datasets for the teacher and the best student per family.
auc_means = summary.set_index("model")["auc_mean"]
print("\n=== Mean ROC AUC across all classification datasets ===")
for m in MODEL_NAMES:
    print(f"{m:14s}: {auc_means[m]:.4f}")

/var/folders/mb/bntbqz296619tr5gmcdrs73w0000gn/T/ipykernel_50908/3593573128.py:165: RuntimeWarning: Mean of empty slice
  row[f"{m}_acc"] = float(np.nanmean(res[m]["acc"]))
/var/folders/mb/bntbqz296619tr5gmcdrs73w0000gn/T/ipykernel_50908/3593573128.py:166: RuntimeWarning: Mean of empty slice
  row[f"{m}_auc"] = float(np.nanmean(res[m]["auc"]))


tic-tac-toe              (k=2) tabpfn=0.627 | rf_base=0.642 | rf_aug=0.627 | rf_aug_conf=nan | mlp_base=0.618 | mlp_aug=0.609 | mlp_aug_conf=nan | mlp_soft=0.558


/var/folders/mb/bntbqz296619tr5gmcdrs73w0000gn/T/ipykernel_50908/3593573128.py:165: RuntimeWarning: Mean of empty slice
  row[f"{m}_acc"] = float(np.nanmean(res[m]["acc"]))
/var/folders/mb/bntbqz296619tr5gmcdrs73w0000gn/T/ipykernel_50908/3593573128.py:166: RuntimeWarning: Mean of empty slice
  row[f"{m}_auc"] = float(np.nanmean(res[m]["auc"]))


monks-problems-2         (k=2) tabpfn=0.623 | rf_base=0.632 | rf_aug=0.611 | rf_aug_conf=nan | mlp_base=0.621 | mlp_aug=0.607 | mlp_aug_conf=nan | mlp_soft=0.559


/var/folders/mb/bntbqz296619tr5gmcdrs73w0000gn/T/ipykernel_50908/3593573128.py:165: RuntimeWarning: Mean of empty slice
  row[f"{m}_acc"] = float(np.nanmean(res[m]["acc"]))
/var/folders/mb/bntbqz296619tr5gmcdrs73w0000gn/T/ipykernel_50908/3593573128.py:166: RuntimeWarning: Mean of empty slice
  row[f"{m}_auc"] = float(np.nanmean(res[m]["auc"]))


...

kc2                      (k=2) tabpfn=0.819 | rf_base=0.827 | rf_aug=0.831 | rf_aug_conf=nan | mlp_base=0.747 | mlp_aug=0.573 | mlp_aug_conf=nan | mlp_soft=0.513
[skip] megawatt1: index 178 is out of bounds for axis 0 with size 1

=== Aggregate across all classification datasets ===


/Users/filipzdebik/Documents/Programming/tabpfn-distill/TabPFN-Distil/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:1775: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


,model,acc_mean,acc_std,auc_mean,auc_std
0,tabpfn,0.662,0.127,0.698,0.148
1,rf_base,0.677,0.119,0.695,0.142
2,rf_aug,0.676,0.117,0.680,0.135
3,rf_aug_conf,NaN,NaN,NaN,NaN
4,mlp_base,0.565,0.138,0.566,0.124
5,mlp_aug,0.596,0.137,0.567,0.132
6,mlp_aug_conf,NaN,NaN,NaN,NaN
7,mlp_soft,0.522,0.110,0.545,0.092


,dataset,n_rows,n_classes,tabpfn_acc,tabpfn_auc,rf_base_acc,rf_base_auc,rf_aug_acc,rf_aug_auc,rf_aug_conf_acc,rf_aug_conf_auc,mlp_base_acc,mlp_base_auc,mlp_aug_acc,mlp_aug_auc,mlp_aug_conf_acc,mlp_aug_conf_auc,mlp_soft_acc,mlp_soft_auc
0,tic-tac-toe,958,2,0.627,0.533,0.642,0.545,0.627,0.519,NaN,NaN,0.618,0.521,0.609,0.514,NaN,NaN,0.558,0.503
1,monks-problems-2,601,2,0.623,0.512,0.632,0.504,0.611,0.517,NaN,NaN,0.621,0.506,0.607,0.506,NaN,NaN,0.559,0.513
2,sonar,208,2,0.612,0.660,0.630,0.675,0.600,0.655,NaN,NaN,0.573,0.593,0.555,0.627,NaN,NaN,0.571,0.542
3,ionosphere,351,2,0.640,0.857,0.663,0.666,0.648,0.740,NaN,NaN,0.670,0.724,0.686,0.742,NaN,NaN,0.673,0.677
4,vehicle,846,4,0.385,0.672,0.388,0.656,0.388,0.668,NaN,NaN,0.240,0.499,0.238,0.464,NaN,NaN,0.267,NaN
5,wdbc,569,2,0.676,0.929,0.693,0.931,0.815,0.851,NaN,NaN,0.513,0.482,0.715,0.653,NaN,NaN,0.545,0.439
6,diabetes,768,2,0.651,0.666,0.660,0.667,0.666,0.649,NaN,NaN,0.412,0.504,0.624,0.479,NaN,NaN,0.577,0.563
7,ilpd,583,2,0.697,0.589,0.703,0.658,0.702,0.644,NaN,NaN,0.442,0.525,0.665,0.646,NaN,NaN,0.504,0.443
8,balance-scale,625,3,0.634,0.674,0.662,0.699,0.610,0.662,NaN,NaN,0.771,0.773,0.738,0.748,NaN,NaN,0.486,NaN
9,blood-transfusion,748,2,0.753,0.667,0.739,0.660,0.708,0.596,NaN,NaN,0.448,0.501,0.552,0.433,NaN,NaN,0.524,0.704



=== Mean ROC AUC across all classification datasets ===
tabpfn        : 0.6980
rf_base       : 0.6951
rf_aug        : 0.6801
rf_aug_conf   : nan
mlp_base      : 0.5661
mlp_aug       : 0.5673
mlp_aug_conf  : nan
mlp_soft      : 0.5451


In [27]:
summary.to_csv("results/synthetic_labels_4_sum.csv")
per_dataset.to_csv("results/synthetic_labels_4_dataset.csv")